# Ames Housing: synthetic smoke-эксперимент

Исполненный автономный эксперимент на **детерминированных синтетических smoke-данных**.
Он проверяет реальный код репозитория, но не оценивает качество на исходном публичном наборе.

## tl;dr

Ниже показан фактически исполненный smoke-run: объём синтетики, выбранная на validation
модель и метрики неизменяемого synthetic test split. Эти числа нельзя переносить на реальные данные.

In [1]:
import io
import json
import sys
import tempfile
from contextlib import redirect_stdout
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ames_housing.data import TARGET, split_data, validate_frame
from ames_housing.evaluate import main as evaluate_model
from ames_housing.generate_smoke_data import generate_smoke_frame
from ames_housing.train import main as train_model

SEED = 20250621
ROWS = 360
smoke_frame = generate_smoke_frame(rows=ROWS, seed=SEED)
validated_frame = validate_frame(smoke_frame)
train_frame, validation_frame, test_frame = split_data(validated_frame, seed=SEED)

temporary_directory = tempfile.TemporaryDirectory(prefix="ames-smoke-")
run_directory = Path(temporary_directory.name)
data_path = run_directory / "smoke.csv"
artifact_path = run_directory / "model.joblib"
validation_path = run_directory / "validation.json"
metrics_path = run_directory / "test_metrics.json"
errors_path = run_directory / "largest_errors.csv"
validated_frame.to_csv(data_path, index=False)

with redirect_stdout(io.StringIO()):
    train_model([
        "--data", str(data_path), "--artifact", str(artifact_path),
        "--report", str(validation_path), "--seed", str(SEED),
    ])
    evaluate_model([
        "--data", str(data_path), "--artifact", str(artifact_path),
        "--metrics", str(metrics_path), "--errors", str(errors_path),
    ])

validation_report = json.loads(validation_path.read_text(encoding="utf-8"))
test_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
error_rows = pd.read_csv(errors_path)
summary = pd.DataFrame([{
    "data": "deterministic synthetic smoke",
    "rows": len(validated_frame),
    "selected_model": test_metrics["model_name"],
    "test_mae": test_metrics["mae"],
    "test_rmse": test_metrics["rmse"],
    "test_r2": test_metrics["r2"],
    "reviewed_largest_errors": len(error_rows),
}])
summary.round(4)

,data,rows,selected_model,test_mae,test_rmse,test_r2,reviewed_largest_errors
0,deterministic synthetic smoke,360,lasso_log_target,22497.7289,27197.4324,0.8719,25


## Context & Methods


Цель — воспроизводимо выполнить реальный регрессионный workflow Ames на автономной синтетике.
`ames_housing.train.main` сравнивает Dummy, линейные модели и ансамбли, проводит CV только на train
и выбирает модель по validation MAE. `evaluate.main` открывает synthetic test и строит анализ остатков.


            ### Key Assumptions


- Seed `20250621`, 360 строк; цены и зависимости сгенерированы и не описывают рынок Ames.
- Разбиение фиксировано 60/20/20; preprocessing обучается только внутри train pipeline.
- Границы дешёвого, среднего и дорогого сегмента вычисляются только на train.
- Денежные единицы smoke-цели условны; MAE/RMSE нельзя сравнивать с реальным OpenML-набором.

## Data

Smoke-таблица создаётся локальным генератором с фиксированным seed, затем проходит ту же
проверку схемы и то же разбиение, что и пользовательский CSV. Сетевые источники не используются.

In [2]:
display(pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_frame), len(validation_frame), len(test_frame)],
    "median_sale_price": [
        train_frame[TARGET].median(), validation_frame[TARGET].median(), test_frame[TARGET].median()
    ],
}).round(2))
display(validated_frame.head(3))

,split,rows,median_sale_price
0,train,216,312619.86
1,validation,72,315191.41
2,test,72,286899.03


,OverallQual,GrLivArea,TotalBsmtSF,GarageCars,YearBuilt,FullBath,LotArea,Neighborhood,HouseStyle,CentralAir,KitchenQual,SaleCondition,SalePrice
0,3,1532.0,689.0,3,1922,2,7285.0,CollgCr,1.5Fin,Y,Gd,Normal,253938.10
1,7,1172.0,526.0,2,1994,1,11031.0,CollgCr,1.5Fin,Y,Gd,Normal,353027.80
2,8,1859.0,1187.0,2,1963,3,7185.0,NridgHt,1Story,Y,TA,Abnorml,434111.43


## Results

Первая таблица — сравнение кандидатов на validation. Следующие результаты относятся только
к synthetic test split; таблица ошибок ограничена несколькими строками.

In [3]:
validation_table = pd.DataFrame(validation_report["models"]).T
display(validation_table[["mae", "rmse", "r2", "cv_mae_mean", "cv_mae_std"]]
        .sort_values("mae").round(2))
display(pd.DataFrame({"metric": ["mae", "rmse", "r2"],
                      "synthetic_test": [test_metrics[key] for key in ["mae", "rmse", "r2"]]}).round(4))
display(pd.DataFrame.from_dict(test_metrics["mae_by_price_band"], orient="index",
                               columns=["synthetic_test_mae"]).round(2))

,mae,rmse,r2,cv_mae_mean,cv_mae_std
lasso_log_target,19283.23,26324.15,0.88,21476.70,924.85
ridge_log_target,19389.49,26062.18,0.88,21440.58,734.02
linear_regression_log_target,19487.41,26560.12,0.87,21605.97,888.14
gradient_boosting,26310.10,33819.29,0.80,27911.92,1219.93
random_forest,29432.25,37653.04,0.75,31073.35,1344.76
dummy_median,63149.27,74981.04,-0.00,61775.70,3790.47


,metric,synthetic_test
0,mae,22497.7289
1,rmse,27197.4324
2,r2,0.8719


,synthetic_test_mae
cheap,21376.43
middle,21892.59
expensive,25087.38


In [4]:
error_rows.head(8)

,Neighborhood,OverallQual,GrLivArea,SalePrice,prediction,residual,absolute_error,price_band
0,Edwards,2,1024.0,132231.87,196788.316937,-64556.446937,64556.446937,cheap
1,Edwards,7,2030.0,372690.92,311023.780094,61667.139906,61667.139906,expensive
2,Edwards,10,1622.0,354004.29,412066.385717,-58062.095717,58062.095717,expensive
3,OldTown,10,2058.0,386136.40,435709.400771,-49573.000771,49573.000771,expensive
4,Edwards,3,2645.0,230331.11,275524.104373,-45192.994373,45192.994373,cheap
5,CollgCr,9,867.0,331216.47,287905.149409,43311.320591,43311.320591,middle
6,NridgHt,9,2505.0,450312.10,493384.302760,-43072.202760,43072.202760,expensive
7,NridgHt,3,1611.0,337836.84,296894.932824,40941.907176,40941.907176,middle


## Takeaways

Выводы ниже сформированы из сохранённых outputs текущего запуска и относятся только к smoke-проверке.

In [5]:
print(f"- На synthetic test выбран {test_metrics['model_name']}: "
      f"MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}, R²={test_metrics['r2']:.4f}.")
print(f"- Сохранён bounded-разбор {len(error_rows)} крупнейших synthetic-остатков и MAE по ценовым полосам.")
print("- Эти условные денежные ошибки подтверждают только исполнение pipeline, не качество на Ames Housing.")

- На synthetic test выбран lasso_log_target: MAE=22497.73, RMSE=27197.43, R²=0.8719.
- Сохранён bounded-разбор 25 крупнейших synthetic-остатков и MAE по ценовым полосам.
- Эти условные денежные ошибки подтверждают только исполнение pipeline, не качество на Ames Housing.
